# Attention Is All You Need — Transformer (from scratch)

Runs on Kaggle without local training load.

In [ ]:
import math
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cpu')
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    if major >= 7:
        device = torch.device('cuda')
print('Device:', device)

In [ ]:
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
VOCAB_SIZE = 128

def generate_pair(min_len=5, max_len=20):
    length = random.randint(min_len, max_len)
    tokens = [random.randint(3, VOCAB_SIZE - 1) for _ in range(length)]
    return tokens, list(reversed(tokens))

class ReverseDataset(Dataset):
    def __init__(self, n_samples):
        self.samples = [generate_pair() for _ in range(n_samples)]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        src, tgt = self.samples[idx]
        src = [BOS_ID] + src + [EOS_ID]
        tgt_in = [BOS_ID] + tgt
        tgt_out = tgt + [EOS_ID]
        return torch.tensor(src), torch.tensor(tgt_in), torch.tensor(tgt_out)

def collate_batch(batch):
    src, tgt_in, tgt_out = zip(*batch)
    src = nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=PAD_ID)
    tgt_in = nn.utils.rnn.pad_sequence(tgt_in, batch_first=True, padding_value=PAD_ID)
    tgt_out = nn.utils.rnn.pad_sequence(tgt_out, batch_first=True, padding_value=PAD_ID)
    return src, tgt_in, tgt_out

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        bsz = q.size(0)
        Q = self.q_proj(q).view(bsz, -1, self.n_heads, self.d_head).transpose(1, 2)
        K = self.k_proj(k).view(bsz, -1, self.n_heads, self.d_head).transpose(1, 2)
        V = self.v_proj(v).view(bsz, -1, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context = attn @ V
        context = context.transpose(1, 2).contiguous().view(bsz, -1, self.d_model)
        return self.o_proj(context)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.ln1(x + self.dropout(self.self_attn(x, x, x, src_mask)))
        x = self.ln2(x + self.dropout(self.ff(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask, src_mask):
        x = self.ln1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.ln2(x + self.dropout(self.cross_attn(x, memory, memory, src_mask)))
        x = self.ln3(x + self.dropout(self.ff(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=192, n_heads=6, n_layers=3, d_ff=768, dropout=0.1):
        super().__init__()
        self.src_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.tgt_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos = PositionalEncoding(d_model)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def make_src_mask(self, src):
        return (src != PAD_ID).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        pad_mask = (tgt != PAD_ID).unsqueeze(1).unsqueeze(2)
        seq_len = tgt.size(1)
        causal = torch.tril(torch.ones((seq_len, seq_len), device=tgt.device)).bool()
        return pad_mask & causal.unsqueeze(0).unsqueeze(1)

    def forward(self, src, tgt_in):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt_in)
        src_x = self.dropout(self.pos(self.src_embed(src)))
        tgt_x = self.dropout(self.pos(self.tgt_embed(tgt_in)))
        memory = src_x
        for layer in self.enc_layers:
            memory = layer(memory, src_mask)
        x = tgt_x
        for layer in self.dec_layers:
            x = layer(x, memory, tgt_mask, src_mask)
        return self.out(x)

In [ ]:
@dataclass
class TrainConfig:
    train_samples: int = 8000
    val_samples: int = 1200
    batch_size: int = 64
    epochs: int = 4
    lr: float = 3e-4

cfg = TrainConfig()
train_loader = DataLoader(ReverseDataset(cfg.train_samples), batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(ReverseDataset(cfg.val_samples), batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_batch)
model = Transformer(vocab_size=VOCAB_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    it = tqdm(loader, leave=False)
    for src, tgt_in, tgt_out in it:
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(src, tgt_in)
        loss = criterion(logits.view(-1, VOCAB_SIZE), tgt_out.view(-1))
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item()
        it.set_description(f'loss={loss.item():.4f}')
    return total_loss / len(loader)

for epoch in range(1, cfg.epochs + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    print(f'epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={va:.4f}')

In [ ]:
@torch.no_grad()
def greedy_decode(src_tokens, max_len=64):
    model.eval()
    src = torch.tensor([BOS_ID] + src_tokens + [EOS_ID], device=device).unsqueeze(0)
    generated = torch.tensor([[BOS_ID]], device=device)
    for _ in range(max_len):
        logits = model(src, generated)
        next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_tok], dim=1)
        if next_tok.item() == EOS_ID:
            break
    return generated.squeeze(0).tolist()

example_src, _ = generate_pair()
pred = greedy_decode(example_src)
print('src:', example_src)
print('target:', [BOS_ID] + list(reversed(example_src)) + [EOS_ID])
print('pred:', pred)

In [ ]:
save_path = '/kaggle/working/transformer_attention_is_all_you_need.pt'
torch.save(model.state_dict(), save_path)
print('saved:', save_path)